<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z346_DTW_Escalado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTW + Normalización — benchmark con desplazamiento temporal y lanzamientos

Extensión de z343. Agrega:

1. **Desplazamiento temporal**: mismo shape pero corrido t+1, t+2, t+3 períodos
2. **Ceros aleatorios**: cada serie tiene sus propios ceros (no las mismas posiciones)
3. **Producto lanzamiento**: impulso inicial + decay, distintos momentos de nacimiento
4. **DTW clustering**: hierarchical clustering con distancia DTW, barrido de ventana (Sakoe-Chiba)
5. **7 normalizaciones**: max, min-max, last, first, mean, z-score, sin normalizar

## ¿Por qué DTW?

La distancia euclídea (y coseno) exige que los picos estén en el mismo mes. Si dos series son idénticas pero una está corrida 2 meses, la distancia euclídea es enorme aunque la forma sea la misma. DTW "estira" el eje temporal para alinear las formas antes de medir la distancia.

In [ ]:
!pip install dtaidistance -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cosine, squareform
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from dtaidistance import dtw, dtw_ndim
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

T          = 36    # meses totales del dataset
K          = 8     # variantes por shape
MAX_SHIFT  = 4     # máximo desplazamiento temporal (períodos)
PROB_CERO  = 0.08  # probabilidad de cero en cualquier mes

print('OK')

# 1. Shapes base — incluyendo curva de lanzamiento

In [ ]:
t = np.arange(T)

def shape_lanzamiento(inicio=0, duracion_impulso=4, altura_impulso=6, nivel_base=1.0):
    """
    Producto nuevo: arranca en `inicio`, tiene impulso inicial de `duracion_impulso` meses
    y luego decae a `nivel_base`.
    """
    s = np.ones(T) * nivel_base
    for i in range(duracion_impulso):
        if inicio + i < T:
            s[inicio + i] = nivel_base + altura_impulso * np.exp(-0.5 * i)
    # antes del nacimiento: 0 (no existía)
    s[:inicio] = 0.0
    return s + np.random.normal(0, 0.05 * nivel_base, T)

SHAPES = {
    'tendencia+':   lambda: np.maximum(1 + t/T*3 + np.random.normal(0, 0.08, T), 0),
    'tendencia-':   lambda: np.maximum(4 - t/T*3 + np.random.normal(0, 0.08, T), 0),
    'estable':      lambda: np.maximum(2 + np.random.normal(0, 0.1, T), 0),
    'estacional':   lambda: np.maximum(1.5 + np.sin(2*np.pi*t/12) + 0.4*np.sin(2*np.pi*t/6) + np.random.normal(0, 0.06, T), 0),
    'lanzamiento':  lambda: shape_lanzamiento(inicio=0, duracion_impulso=5, altura_impulso=5, nivel_base=1.0),
    'V':            lambda: np.maximum(np.concatenate([np.linspace(3,0.3,T//2), np.linspace(0.3,3,T-T//2)]) + np.random.normal(0, 0.06, T), 0),
    'escalon':      lambda: np.maximum(np.array([1.0 if i < T//2 else 3.5 for i in range(T)]) + np.random.normal(0, 0.06, T), 0),
}
N_SHAPES      = len(SHAPES)
shape_nombres = list(SHAPES.keys())

# Visualizar shapes base
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for i, (nombre, fn) in enumerate(SHAPES.items()):
    np.random.seed(i)
    axes[i].plot(fn(), 'o-', linewidth=1.5, markersize=3)
    axes[i].set_title(nombre, fontsize=10)
    axes[i].set_ylim(bottom=0)
    axes[i].grid(alpha=0.3)
axes[-1].set_visible(False)
fig.suptitle('Shapes base (escala=1, sin perturbaciones)', fontsize=12)
plt.tight_layout()
plt.show()

# 2. Generar dataset — escala aleatoria + desplazamiento + ceros propios

In [ ]:
def generar_serie(fn, escala, shift, rng):
    """
    Genera una variante de una serie con:
    - escala multiplicativa aleatoria
    - desplazamiento temporal (shift períodos)
    - ceros aleatorios propios (distintos para cada serie)
    """
    np.random.seed(rng.integers(0, 99999))
    base = fn() * escala

    # desplazar: la serie empieza `shift` meses después
    if shift > 0:
        s = np.concatenate([np.zeros(shift), base[:-shift]])
    else:
        s = base.copy()

    # ceros aleatorios propios de esta serie
    mask_cero = rng.random(T) < PROB_CERO
    # no poner ceros donde ya hay 0 (inicio de serie nueva)
    primer_nonzero = next((i for i, v in enumerate(s) if v > 0), T)
    mask_cero[:primer_nonzero] = False
    s[mask_cero] = 0.0

    return s, shift


registros = []
rng = np.random.default_rng(77)

for shape_idx, (nombre_shape, fn) in enumerate(SHAPES.items()):
    # para 'lanzamiento': distintos momentos de inicio
    for k in range(K):
        escala = float(np.exp(rng.uniform(np.log(0.1), np.log(150))))

        if nombre_shape == 'lanzamiento':
            # cada producto nace en un momento distinto
            shift = int(rng.integers(0, T // 2))
            fn_k  = lambda shift=shift: shape_lanzamiento(
                inicio=0, duracion_impulso=5, altura_impulso=5, nivel_base=1.0
            )
        else:
            # desplazamiento pequeño para los otros shapes
            shift = int(rng.integers(0, MAX_SHIFT + 1))
            fn_k  = fn

        serie, sh = generar_serie(fn_k, escala, shift, rng)

        registros.append({
            'id':          f'{nombre_shape}_{k:02d}',
            'shape':       nombre_shape,
            'shape_idx':   shape_idx,
            'k':           k,
            'escala_real': escala,
            'shift':       sh,
            'serie':       serie,
        })

labels_reales = np.array([r['shape_idx'] for r in registros])
print(f'{len(registros)} series  |  {N_SHAPES} shapes × {K} variantes')
print(f'Escalas: {min(r["escala_real"] for r in registros):.3f} – {max(r["escala_real"] for r in registros):.1f}')
print(f'Shifts: 0–{MAX_SHIFT} meses para shapes normales; 0–{T//2} para lanzamiento')

In [ ]:
# Visualizar el shape 'lanzamiento' — todos en distinto momento
lanz = [r for r in registros if r['shape'] == 'lanzamiento']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for r in lanz:
    ax.plot(r['serie'], alpha=0.7, linewidth=1.5, label=f"shift={r['shift']} ×{r['escala_real']:.1f}")
ax.set_title('Lanzamiento — distinto momento de nacimiento y escala', fontsize=10)
ax.legend(fontsize=6, ncol=2)
ax.grid(alpha=0.3)

ax = axes[1]
for r in lanz:
    s = r['serie']
    m = s.max()
    ax.plot(s / m if m > 0 else s, alpha=0.7, linewidth=1.5)
ax.set_title('Lanzamiento — normalizadas por max (¿se superponen?)', fontsize=10)
ax.grid(alpha=0.3)

plt.suptitle('El desafío: mismo patrón impulso+decay, distinto momento de nacimiento', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar desplazamiento temporal — shape estacional
estac = [r for r in registros if r['shape'] == 'estacional']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
for r in estac[:5]:
    ax.plot(r['serie'], alpha=0.75, linewidth=1.5, label=f"shift={r['shift']}")
ax.set_title('Estacional — distintas escalas y shifts', fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
for r in estac[:5]:
    s = r['serie']; m = s.max()
    ax.plot(s / m if m > 0 else s, alpha=0.75, linewidth=1.5, label=f"shift={r['shift']}")
ax.set_title('Estacional normalizada por max — los picos NO coinciden', fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Problema del desplazamiento: norm_max no alinea los picos', fontsize=11)
plt.tight_layout()
plt.show()

# 3. Normalizaciones — 7 variantes

In [ ]:
def safe_div(s, denom):
    return s / denom if denom != 0 else s.copy()

def norm_sin(s):      return s.copy(), 1.0

def norm_max(s):
    pos = s[s > 0]; m = float(pos.max()) if len(pos) > 0 else 1.0
    return safe_div(s, m), m

def norm_minmax(s):
    pos = s[s > 0]
    mn  = float(pos.min()) if len(pos) > 0 else 0.0
    mx  = float(pos.max()) if len(pos) > 0 else 1.0
    rng = mx - mn
    if rng == 0: return s.copy(), (mn, 1.0)
    return (s - mn) / rng, (mn, rng)

def norm_last(s, ventana=3):
    pos   = s[s > 0]
    ultimos = s[-ventana:]; ultimos = ultimos[ultimos > 0]
    base  = float(ultimos.mean()) if len(ultimos) > 0 else (float(pos.mean()) if len(pos) > 0 else 1.0)
    return safe_div(s, base), base

def norm_first(s, ventana=3):
    pos   = s[s > 0]
    primeros = pos[:ventana]
    base  = float(primeros.mean()) if len(primeros) > 0 else 1.0
    return safe_div(s, base), base

def norm_mean(s):
    pos  = s[s > 0]; base = float(pos.mean()) if len(pos) > 0 else 1.0
    return safe_div(s, base), base

def norm_zscore(s):
    pos   = s[s > 0]
    mu    = float(pos.mean()) if len(pos) > 0 else 0.0
    sigma = float(pos.std())  if len(pos) > 1 else 1.0
    if sigma == 0: sigma = 1.0
    n = (s - mu) / sigma
    mn = n[s > 0].min() if (s > 0).any() else 0.0
    if mn < 0: n[s >= 0] -= mn
    return n, (mu, sigma)

NORMALIZACIONES = {
    'sin_norm': norm_sin,
    'max':      norm_max,
    'min_max':  norm_minmax,
    'last':     norm_last,
    'first':    norm_first,
    'mean':     norm_mean,
    'zscore':   norm_zscore,
}
print('Normalizaciones:', list(NORMALIZACIONES.keys()))

In [ ]:
# Ver superposición de variantes estacional por normalización
variantes_test = [r for r in registros if r['shape'] == 'estacional'][:5]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, (nombre_norm, fn) in enumerate(NORMALIZACIONES.items()):
    ax = axes[i]
    for r in variantes_test:
        ax.plot(fn(r['serie'])[0], alpha=0.8, linewidth=1.5)
    ax.set_title(f'norm={nombre_norm}', fontsize=9)
    ax.grid(alpha=0.3)
axes[-1].set_visible(False)
fig.suptitle('Estacional — 5 variantes (distintas escalas + shifts) por normalización', fontsize=10)
plt.tight_layout()
plt.show()

# 4. DTW — distancia y clustering

DTW permite comparar series desplazadas temporalmente. El parámetro `window` (banda de Sakoe-Chiba) limita cuántos meses puede "estirarse" el alineamiento:
- `window=None` → DTW libre (puede alinear cualquier par de meses)
- `window=3` → solo puede desplazar ±3 meses
- `window=1` → casi igual a distancia euclídea

In [ ]:
def calcular_matriz_dtw(registros, norm_fn, window=None):
    """Calcula la matriz de distancias DTW entre todas las series normalizadas."""
    series = []
    for r in registros:
        s_norm, _ = norm_fn(r['serie'])
        series.append(s_norm.astype(np.double))

    kwargs = {'window': window} if window is not None else {}
    mat = dtw.distance_matrix_fast(series, **kwargs)
    # la diagonal puede tener nan en algunos builds
    np.fill_diagonal(mat, 0)
    return mat


def calcular_matriz_euclidea(registros, norm_fn):
    """Matriz de distancias euclídeas para comparar con DTW."""
    from scipy.spatial.distance import cdist
    X = np.array([norm_fn(r['serie'])[0] for r in registros])
    return cdist(X, X, metric='euclidean')


def ari_desde_matriz(mat_dist, n_clusters=N_SHAPES):
    """Clustering jerárquico sobre matriz de distancias → ARI."""
    condensada = squareform(mat_dist, checks=False)
    Z = linkage(condensada, method='average')
    labels_pred = fcluster(Z, n_clusters, criterion='maxclust')
    return adjusted_rand_score(labels_reales, labels_pred), Z


print('Funciones DTW listas.')

# 5. Barrido — normalizaciones × ventanas DTW

In [ ]:
WINDOWS_DTW = [None, 2, 4, 6, 12]   # None = DTW libre

resultados_dtw = {}   # (norm, window) → ARI
resultados_euc = {}   # norm → ARI euclidean

print(f'{"NORM":10s}  {"METRICA":20s}  {"ARI":>6s}')
print('-' * 42)

for nombre_norm, fn in NORMALIZACIONES.items():
    # euclidean baseline
    mat_euc = calcular_matriz_euclidea(registros, fn)
    ari_euc, _ = ari_desde_matriz(mat_euc)
    resultados_euc[nombre_norm] = ari_euc
    print(f'{nombre_norm:10s}  {"euclidean":20s}  {ari_euc:6.4f}')

    # DTW con distintas ventanas
    for window in WINDOWS_DTW:
        label = f'dtw_w{window}' if window else 'dtw_libre'
        mat_dtw = calcular_matriz_dtw(registros, fn, window=window)
        ari, _ = ari_desde_matriz(mat_dtw)
        resultados_dtw[(nombre_norm, window)] = ari
        print(f'{nombre_norm:10s}  {label:20s}  {ari:6.4f}')
    print()

In [ ]:
# Heatmap ARI: normalizaciones × ventanas DTW
norms_list   = list(NORMALIZACIONES.keys())
windows_list = [None, 2, 4, 6, 12]
window_labels = ['libre', 'w=2', 'w=4', 'w=6', 'w=12']

mat_ari = np.zeros((len(norms_list), len(windows_list)))
for i, n in enumerate(norms_list):
    for j, w in enumerate(windows_list):
        mat_ari[i, j] = resultados_dtw[(n, w)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# heatmap DTW
ax = axes[0]
im = ax.imshow(mat_ari, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(windows_list))); ax.set_xticklabels(window_labels, fontsize=9)
ax.set_yticks(range(len(norms_list)));   ax.set_yticklabels(norms_list, fontsize=9)
ax.set_xlabel('Ventana DTW (Sakoe-Chiba)')
ax.set_title('ARI clustering DTW\n(normalización × ventana)', fontsize=10)
plt.colorbar(im, ax=ax)
for i in range(len(norms_list)):
    for j in range(len(windows_list)):
        ax.text(j, i, f'{mat_ari[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='black' if mat_ari[i,j] > 0.3 else 'white')

# barras euclidean vs mejor DTW por normalización
ax = axes[1]
x = np.arange(len(norms_list))
width = 0.35
ari_euc_vals = [resultados_euc[n] for n in norms_list]
ari_dtw_best = [mat_ari[i].max() for i in range(len(norms_list))]
ax.bar(x - width/2, ari_euc_vals, width, label='Euclidean', color='steelblue', alpha=0.8)
ax.bar(x + width/2, ari_dtw_best, width, label='DTW (mejor ventana)', color='tomato', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(norms_list, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('ARI'); ax.set_ylim(0, 1.05)
ax.set_title('Euclidean vs DTW (mejor ventana) por normalización', fontsize=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.4)
for i, (e, d) in enumerate(zip(ari_euc_vals, ari_dtw_best)):
    ax.text(i - width/2, e + 0.01, f'{e:.2f}', ha='center', fontsize=7)
    ax.text(i + width/2, d + 0.01, f'{d:.2f}', ha='center', fontsize=7)

fig.suptitle('DTW vs Euclidean — impacto de la normalización y la ventana', fontsize=12)
plt.tight_layout()
plt.show()

# 6. DTW ilustrado — alineamiento entre dos series desplazadas

In [ ]:
from dtaidistance import dtw_visualisation as dtwvis

# tomar dos variantes del shape estacional con distinto shift
estac = sorted([r for r in registros if r['shape'] == 'estacional'], key=lambda r: r['shift'])
r1 = estac[0]   # shift pequeño
r2 = next((r for r in estac if r['shift'] >= 2), estac[-1])  # shift mayor

# normalizar con max
s1 = norm_max(r1['serie'])[0].astype(np.double)
s2 = norm_max(r2['serie'])[0].astype(np.double)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# distancia euclidea vs dtw
d_euc = np.sqrt(((s1 - s2)**2).sum())
d_dtw = dtw.distance(s1, s2)

ax = axes[0]
ax.plot(s1, 'o-', color='steelblue', linewidth=1.5, markersize=3, label=f'shift={r1["shift"]} ×{r1["escala_real"]:.1f}')
ax.plot(s2, 'o-', color='tomato',    linewidth=1.5, markersize=3, label=f'shift={r2["shift"]} ×{r2["escala_real"]:.1f}')
ax.set_title(f'Estacional — 2 variantes normalizadas por max\nEuclidean={d_euc:.3f}  |  DTW={d_dtw:.3f}', fontsize=9)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# path DTW
d_mat = dtw.distance_matrix_fast([s1, s2])
path  = dtw.warping_path(s1, s2)
ax = axes[1]
ax.plot(s1, 'o-', color='steelblue', linewidth=1.5, markersize=3, alpha=0.8)
ax.plot(s2, 'o-', color='tomato',    linewidth=1.5, markersize=3, alpha=0.8)
for pi, pj in path[::3]:  # mostrar 1 de cada 3 conexiones para no saturar
    ax.plot([pi, pj], [s1[pi], s2[pj]], 'gray', linewidth=0.5, alpha=0.5)
ax.set_title('Alineamiento DTW — líneas grises conectan meses equivalentes', fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('DTW alinea temporalmente las series antes de medir la distancia', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Distancia euclidean: {d_euc:.4f}')
print(f'Distancia DTW:       {d_dtw:.4f}')
print(f'DTW es {d_euc/d_dtw:.1f}x más chica → estas dos series se ven más parecidas con DTW')

# 7. Foco en 'lanzamiento' — DTW agrupa distintos momentos de nacimiento?

In [ ]:
# Extraer solo los productos lanzamiento + un shape control (estable)
sub = [r for r in registros if r['shape'] in ('lanzamiento', 'estable')]
labels_sub = np.array([r['shape_idx'] for r in sub])

print('ARI sobre subconjunto lanzamiento vs estable:')
print(f'{"METRICA":25s}  {"ARI":>6s}')
print('-' * 35)

for nombre_norm, fn in [('sin_norm', norm_sin), ('max', norm_max), ('first', norm_first)]:
    # euclidean
    from scipy.spatial.distance import cdist
    X = np.array([fn(r['serie'])[0] for r in sub])
    mat_e = cdist(X, X, 'euclidean')
    cond_e = squareform(mat_e, checks=False)
    Z_e = linkage(cond_e, method='average')
    lp_e = fcluster(Z_e, 2, criterion='maxclust')
    ari_e = adjusted_rand_score(labels_sub, lp_e)
    print(f'{nombre_norm + "_euclidean":25s}  {ari_e:6.4f}')

    # DTW libre
    series_norm = [fn(r['serie'])[0].astype(np.double) for r in sub]
    mat_d = dtw.distance_matrix_fast(series_norm)
    np.fill_diagonal(mat_d, 0)
    cond_d = squareform(mat_d, checks=False)
    Z_d = linkage(cond_d, method='average')
    lp_d = fcluster(Z_d, 2, criterion='maxclust')
    ari_d = adjusted_rand_score(labels_sub, lp_d)
    print(f'{nombre_norm + "_DTW_libre":25s}  {ari_d:6.4f}')
print()
print('Si DTW > euclidean → DTW reconoce el patrón impulso+decay aunque nazcan en distinto momento.')

# 8. Dendrograma — mejor normalización × mejor ventana DTW

In [ ]:
# Mejor combinación
mejor_key = max(resultados_dtw, key=lambda k: resultados_dtw[k])
mejor_norm, mejor_window = mejor_key
mejor_ari = resultados_dtw[mejor_key]
print(f'Mejor: norm={mejor_norm}, DTW window={mejor_window}, ARI={mejor_ari:.4f}')

fn_mejor = NORMALIZACIONES[mejor_norm]
mat_mejor = calcular_matriz_dtw(registros, fn_mejor, window=mejor_window)
_, Z_mejor = ari_desde_matriz(mat_mejor)

COLORES = plt.cm.tab10(np.linspace(0, 1, N_SHAPES))
labels_plot = [f"{r['shape'][:5]}_{r['k']:02d}" for r in registros]

fig, ax = plt.subplots(figsize=(20, 7))
dendrogram(Z_mejor, labels=labels_plot, ax=ax, leaf_rotation=90, leaf_font_size=7, color_threshold=0)
xlbls = ax.get_xmajorticklabels()
for lbl in xlbls:
    idx = next((i for i, r in enumerate(registros) if f"{r['shape'][:5]}_{r['k']:02d}" == lbl.get_text()), 0)
    lbl.set_color(COLORES[registros[idx]['shape_idx']])

handles = [plt.Line2D([0],[0], color=COLORES[i], linewidth=3, label=s) for i, s in enumerate(shape_nombres)]
ax.legend(handles=handles, fontsize=8, loc='upper right')
ax.set_title(f'Dendrograma DTW — norm={mejor_norm}, window={mejor_window}, ARI={mejor_ari:.4f}\n(colores = shape real)', fontsize=11)
ax.set_ylabel('distancia DTW')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 9. Resumen — tabla completa

In [ ]:
print('=' * 65)
print(f'  {"NORM":10s}  {"Euclidean":>10s}  ', end='')
for w in windows_list:
    lbl = f'dtw_{w}' if w else 'dtw_libre'
    print(f'{lbl:>10s}  ', end='')
print()
print('=' * 65)
for nombre_norm in norms_list:
    print(f'  {nombre_norm:10s}  {resultados_euc[nombre_norm]:>10.4f}  ', end='')
    for w in windows_list:
        print(f'{resultados_dtw[(nombre_norm, w)]:>10.4f}  ', end='')
    print()
print('=' * 65)
print()
print(f'Mejor combinación: norm={mejor_norm}, DTW window={mejor_window}, ARI={mejor_ari:.4f}')
print()
print('Interpretación ventana DTW:')
print('  libre → puede alinear cualquier par de meses (máxima flexibilidad)')
print('  w=2   → solo desplaza ±2 meses (útil si los shifts son pequeños)')
print('  w=12  → puede absorber desfasajes de hasta 1 año')
print()
print('Ventana muy grande puede perjudicar: une shapes distintos si son globalmente parecidos')